    # Week 10 · Python Transport Networks
    
    Apply network analysis workflows using OSMnx and related libraries to mirror QGIS service area studies.
    


    ## Learning goals
    
    - Download and prepare street networks with OSMnx.
    - Compute travel-time isochrones around key facilities.
    - Summarise population coverage or equity metrics using joined census data.
    


    ## 1. Imports & configuration
    


In [ ]:
    from pathlib import Path
    
    import geopandas as gpd
    import networkx as nx
    import osmnx as ox
    
    ox.settings.log_console = True
    ox.settings.use_cache = True
    
    DATA_ROOT = Path("..") / "data" / "processed" / "week10"
    FACILITIES_PATH = DATA_ROOT / "facilities.geojson"
    if not FACILITIES_PATH.exists():
        raise FileNotFoundError("Provide facility locations at data/processed/week10/facilities.geojson")
    
    facilities = gpd.read_file(FACILITIES_PATH).to_crs(4326)
    facilities.head()
    


    ## 2. Download network
    


In [ ]:
    # Update place query or boundary geometry to your study area
    PLACE = "Boston, Massachusetts, USA"
    G = ox.graph_from_place(PLACE, network_type="walk")
    print(nx.info(G))
    


    ## 3. Build isochrones
    


In [ ]:
    import pandas as pd
    
    TRAVEL_TIMES = [5, 10, 15]  # minutes
    SPEED_KMPH = 4.8  # walking speed
    METERS_PER_MIN = SPEED_KMPH * 1000 / 60
    
    nodes, edges = ox.graph_to_gdfs(G)
    facilities_proj = facilities.to_crs(edges.crs)
    isochrones = []
    
    for _, facility in facilities_proj.iterrows():
        center_node = ox.distance.nearest_nodes(G, facility.geometry.x, facility.geometry.y)
        lengths = nx.single_source_dijkstra_path_length(G, center_node, cutoff=max(TRAVEL_TIMES) * METERS_PER_MIN, weight="length")
        for minutes in TRAVEL_TIMES:
            reachable_nodes = [node for node, length in lengths.items() if length <= minutes * METERS_PER_MIN]
            subgraph = G.subgraph(reachable_nodes)
            polygon = ox.utils_graph.graph_area_polygon(subgraph, buffer_dist=0)
            isochrones.append({
                "facility_id": facility.get("facility_id", _),
                "minutes": minutes,
                "geometry": polygon,
            })
    
    isochrone_gdf = gpd.GeoDataFrame(isochrones, crs=edges.crs)
    isochrone_gdf.head()
    


    ## 4. Coverage analysis
    


In [ ]:
    POP_DATA = DATA_ROOT / "population.geojson"
    if POP_DATA.exists():
        population = gpd.read_file(POP_DATA).to_crs(isochrone_gdf.crs)
        coverage = gpd.overlay(population, isochrone_gdf, how="intersection")
        print(coverage.head())
    else:
        print("Add population polygons to quantify coverage metrics.")
    


    ## 5. Export & reflection
    


In [ ]:
    OUTPUT_DIR = DATA_ROOT / "outputs"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    isochrone_gdf.to_file(OUTPUT_DIR / "isochrones.gpkg", layer="isochrones", driver="GPKG")
    


In [ ]:
    reflection = {
        "network_learning": "What worked well in Python for network analysis?",
        "qgis_integration": "How will you visualise these isochrones back in QGIS?",
        "next_steps": "List enhancements for capstone projects."
    }
    for key, value in reflection.items():
        print(f"{key}: {value}")
    
